In [21]:
import pandas as pd
import numpy as np
import joblib
import dill

df = pd.read_csv('./data/telco_churn_data.csv')
df
df.TotalCharges = df.TotalCharges.str.replace(' ', '0').astype(float)
df.drop('customerID', axis=1, inplace=True)

y = df['Churn']
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [13]:
df.isna().sum()
df.duplicated().sum()


y = df['Churn']
df_object = df.select_dtypes(include='object').drop('Churn',axis=1)
df_numeric = df.select_dtypes(include='number')

df_object.nunique()
df[df['TotalCharges'] == ' ']
df[df['tenure'] == 0]

gt3_features = df_object.nunique()[df_object.nunique() >= 3].index.tolist()
df_object_gt3 = df_object[gt3_features]
df_object_lt3 = df_object.replace({"Yes": 1, "No": 0})

#gt3_features.pop()
gt3_features

df_object2 = df.select_dtypes(include='object').drop('Churn', axis=1)
df_object2


/tmp/ipykernel_192883/1024720189.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_object = df.select_dtypes(include='object').drop('Churn',axis=1)
/tmp/ipykernel_192883/1024720189.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
0,Female,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check
1,Male,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check
2,Male,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check
3,Male,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic)
4,Female,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check
7039,Female,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic)
7040,Female,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check
7041,Male,Yes,No,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder


X_train, X_test, y_train, y_test = train_test_split(df.drop('Churn', axis=1), y, test_size=0.2, random_state=42)


X_train_cat = X_train.select_dtypes(include="string").astype('category').reset_index(drop=True)
X_train_numeric = X_train.select_dtypes(include="number").drop('SeniorCitizen', axis=1).reset_index(drop=True)

ohe = OneHotEncoder(handle_unknown='infrequent_if_exist')
enc = ohe.fit(X_train_cat)
enc.get_feature_names_out()
X_train_cat = pd.DataFrame(ohe.transform(X_train_cat[X_train_cat.columns.tolist()]).toarray(), columns=[col for col in ohe.get_feature_names_out()])

X0_train = pd.concat([X_train_numeric, X_train_cat], axis=1)


with open('./data/encoder_ohe.pkl', 'wb') as file:
    joblib.dump(ohe, file)

lr = LogisticRegression(penalty='l1', solver='liblinear')
dtc = DecisionTreeClassifier(max_depth=7)
rfc = RandomForestClassifier(max_depth=10, n_estimators=100, min_samples_split=3, random_state=42)
gbc = GradientBoostingClassifier(max_depth=7, n_estimators=100, learning_rate=0.05)

lr.fit(X0_train, y_train)
dtc.fit(X0_train, y_train)
rfc.fit(X0_train, y_train)
gbc.fit(X0_train, y_train)

X_test_cat = X_test.select_dtypes(include="string").astype('category').reset_index(drop=True)
X_test_numeric = X_test.select_dtypes(include="number").drop('SeniorCitizen', axis=1).reset_index(drop=True)
X_test_cat = pd.DataFrame(enc.transform(X_test_cat).toarray(), columns=[col for col in enc.get_feature_names_out()])
X0_test = pd.concat([X_test_numeric, X_test_cat], axis=1)

y_pred_lr = lr.predict(X0_test)
y_pred_dtc = dtc.predict(X0_test)
y_pred_rfc = rfc.predict(X0_test)
y_pred_gbc = gbc.predict(X0_test)

y_test = y_test.reset_index(drop=True)

y_pred_lr.shape, y_test.shape

print(f"LogisticRegression Accuracy: {accuracy_score(y_test, y_pred_lr)}")
print(f"DecisionTreeClassifier Accuracy: {accuracy_score(y_test, y_pred_dtc)}")
print(f"RandomForestClassifier Accuracy: {accuracy_score(y_test, y_pred_rfc)}")
print(f"GradientBoostingClassifier Accuracy: {accuracy_score(y_test, y_pred_gbc)}")





/home/merkis/macaw_ml/bootcamp_macaw_04_26/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/merkis/macaw_ml/bootcamp_macaw_04_26/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


LogisticRegression Accuracy: 0.8197303051809794
DecisionTreeClassifier Accuracy: 0.7955997161107168
RandomForestClassifier Accuracy: 0.8097941802696949
GradientBoostingClassifier Accuracy: 0.8019872249822569


In [15]:
joblib.dump(lr, "data/best_model.pkl")

['data/best_model.pkl']